# 02.04 — Enregistrer images + télémétrie synchronisées

Capture des images horodatées du plugin et leur télémétrie proche, sans toucher au volant. Sauvegarde les PNG et les métadonnées dans un dossier de session. **Ne pas lancer en même temps que le PPO ou un autre notebook UDP.**

In [6]:
from pathlib import Path
import sys
CWD=Path.cwd().resolve()
REPO=next((p for p in [CWD,*CWD.parents] if (p/'assetto_corsa_gym'/'assetto_corsa_gym'/'AssettoCorsaPlugin').is_dir()),None)
if REPO is None:
    raise FileNotFoundError('Ouvrir la racine IA_projet_Ouarti_ds_github ou un sous-dossier dans VS Code.')
PROJECT=REPO/'Code'
GYM_ROOT=REPO/'assetto_corsa_gym'/'assetto_corsa_gym'
assert PROJECT.is_dir(), 'Dossier Code introuvable'
if sys.version_info[:2] != (3,9):
    raise RuntimeError('Selectionner le kernel Python 3.9.13 dans VS Code. Actuel : '+sys.version)
for name in ['datasets','models','reports']:
    (PROJECT/name).mkdir(parents=True, exist_ok=True)
print('Depot :',REPO,'\nCode :',PROJECT,'\nPython :',sys.executable)

Depot : C:\Users\elkat\Documents\Github\IA_projet_Ouarti_ds_github 
Code : C:\Users\elkat\Documents\Github\IA_projet_Ouarti_ds_github\Code 
Python : c:\Users\elkat\AppData\Local\Programs\Python\Python39\python.exe


In [7]:
import socket, json, ast, time
HOST='127.0.0.1'; PORT_UDP=2345; PORT_TCP=2347

def management(command,timeout=4):
    if command not in ('get_config','get_static_info'):
        raise ValueError('Cette fonction attend une commande qui retourne des donnees.')
    with socket.create_connection((HOST,PORT_TCP),timeout=timeout) as s:
        s.settimeout(timeout)
        s.sendall(command.encode('ascii'))
        data=b''
        deadline=time.monotonic()+timeout
        while time.monotonic()<deadline and len(data)<1024*1024:
            try: chunk=s.recv(65536)
            except socket.timeout: break
            if not chunk: break
            data+=chunk
            try: return json.loads(data)
            except (ValueError,UnicodeError): pass
            try: return ast.literal_eval(data.decode('utf8'))
            except (ValueError,SyntaxError,UnicodeError): pass
    raise RuntimeError('Reponse management absente/incomplete pour '+command+': '+repr(data[:150]))

def request_reset():
    # Le protocole sensors_par NE renvoie AUCUNE reponse a la commande reset.
    with socket.create_connection((HOST,PORT_TCP),timeout=4) as s:
        s.settimeout(4)
        s.sendall(b'reset')
        # Fermer la connexion : le serveur traite la commande sans ACK.

def open_telemetry(timeout=6):
    sock=socket.socket(socket.AF_INET,socket.SOCK_DGRAM)
    sock.settimeout(.30)
    sock.connect((HOST,PORT_UDP))
    end=time.monotonic()+timeout
    try:
        while time.monotonic()<end:
            sock.send(b'connect')
            try: data=sock.recv(2**18)
            except socket.timeout: continue
            if data==b'identified': return sock
        raise TimeoutError('Le plugin UDP 2345 ne repond pas : lancer une session de conduite.')
    except Exception:
        sock.close()
        raise

def next_packet(sock,timeout=3,after_steps=None):
    deadline=time.monotonic()+timeout
    while time.monotonic()<deadline:
        try: data=sock.recv(2**18)
        except socket.timeout: continue
        if data==b'disconnect': raise ConnectionError('Connexion telemetrie interrompue')
        if data==b'identified': continue
        try: packet=json.loads(data)
        except (ValueError,UnicodeError): continue
        if not isinstance(packet,dict): continue
        if after_steps is not None and int(packet.get('steps',-1))<=int(after_steps): continue
        packet['_received_unix']=time.time()
        return packet
    raise TimeoutError('Aucun nouvel echantillon telemetrie dans le delai imparti.')

def close_telemetry(sock):
    if sock is None: return
    try: sock.send(b'disconnect')
    except OSError: pass
    sock.close()

In [8]:
import sys, time, numpy as np, cv2
PLUGIN_PATH=GYM_ROOT/'AssettoCorsaPlugin'/'plugins'/'sensors_par'
if not PLUGIN_PATH.is_dir():
    raise FileNotFoundError('Plugin Python source manquant : '+str(PLUGIN_PATH))
if str(PLUGIN_PATH) not in sys.path:
    sys.path.insert(0,str(PLUGIN_PATH))
from dual_buffer import GrabberSharedMemoryDualBuffer

class CameraReader:
    def __init__(self,config,age_limit=.55):
        if not config.get('screen_capture_enable'):
            raise RuntimeError('screen_capture_enable=False dans le plugin installe.')
        self.cam=GrabberSharedMemoryDualBuffer(int(config['final_image_width']),
                      int(config['final_image_height']),str(config['color_mode']))
        self.age_limit=float(age_limit)
        self.last_seq=-1
    def read(self,timeout=2,require_new=True):
        deadline=time.monotonic()+float(timeout)
        last_err='Aucune image stable et fraiche'
        while time.monotonic()<deadline:
            try:
                seq,stamp,arr=self.cam.get_timed_image()
                age=time.time()-float(stamp)
                if age<-.15 or age>self.age_limit:
                    last_err='Image perimee ou horloge incoherente (age %.3f s)'%age
                elif require_new and int(seq)<=self.last_seq:
                    last_err='Image non actualisee (seq %s)'%seq
                else:
                    self.last_seq=int(seq)
                    return int(seq),float(stamp),arr
            except (RuntimeError,ValueError) as e:
                last_err=str(e)
            time.sleep(.015)
        raise TimeoutError('Capture indisponible : '+last_err)

def frame_to_image84(frame):
    if frame.ndim==3:
        # La capture couleur du plugin est en BGR.
        gray=cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)
    elif frame.ndim==2:
        gray=frame
    else:
        raise ValueError('Format camera invalide : '+str(frame.shape))
    gray=cv2.resize(gray,(84,84),interpolation=cv2.INTER_AREA)
    return np.ascontiguousarray(gray[np.newaxis,:,:],dtype=np.uint8)

In [9]:
from datetime import datetime
from PIL import Image
import json, time, numpy as np
DUREE_SECONDES=30
MAX_IMAGES=500
TOLERANCE_SYNCHRO_S=.40
LANCER_CAPTURE=True
cfg=management('get_config')
static=management('get_static_info')
print('Cam:',cfg.get('screen_capture_enable'),cfg.get('color_mode'),
      cfg.get('final_image_width'),cfg.get('final_image_height'))
print('Circuit:',static.get('TrackName'),static.get('TrackConfiguration'),'Auto:',static.get('CarName'))

Cam: True gray 320 240
Circuit: ks_barcelona layout_gp Auto: abarth500


In [10]:
if not LANCER_CAPTURE:
    print('Capture DESACTIVEE : mettre LANCER_CAPTURE=True pour lancer.')
else:
    camera=CameraReader(cfg)
    session=PROJECT/'datasets'/('session_'+datetime.now().strftime('%Y%m%d_%H%M%S'))
    (session/'images').mkdir(parents=True,exist_ok=False)
    (session/'session.json').write_text(json.dumps({'type':'camera_telemetry',
            'created_at':datetime.now().isoformat(),'config':cfg,'static':static},
            indent=2,ensure_ascii=False,default=str),encoding='utf8')
    sock=None;count=0;rejected=0;start=time.monotonic()
    try:
        sock=open_telemetry()
        with (session/'frames.jsonl').open('w',encoding='utf8') as out, \
             (session/'telemetry.jsonl').open('w',encoding='utf8') as telem:
            while count<MAX_IMAGES and time.monotonic()-start<DUREE_SECONDES:
                raw=next_packet(sock,timeout=3)
                try:
                    seq,stamp,frame=camera.read(timeout=1.2)
                except TimeoutError:
                    rejected+=1;continue
                telemetry_time=float(raw.get('timestamp_unix',raw['_received_unix']))
                gap=abs(stamp-telemetry_time)
                if gap>TOLERANCE_SYNCHRO_S:
                    rejected+=1;continue
                name='images/frame_%06d.png'%count
                if frame.ndim==3:
                    rgb=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)
                    Image.fromarray(rgb).save(session/name)
                else:
                    Image.fromarray(frame).save(session/name)
                info={'image':name,'camera_sequence':seq,'camera_time':stamp,
                      'telemetry_time':telemetry_time,'camera_gap_s':gap,
                      'speed':raw.get('speed'),'lap':raw.get('LapCount'),
                      'steps':raw.get('steps')}
                out.write(json.dumps(info,ensure_ascii=False)+'\n')
                telem.write(json.dumps(raw,ensure_ascii=False,default=str)+'\n')
                count+=1
    finally:
        close_telemetry(sock)
    print('Images:',count,'- rejetees:',rejected,'\nSession:',session)

Images: 90 - rejetees: 678 
Session: C:\Users\elkat\Documents\Github\IA_projet_Ouarti_ds_github\Code\datasets\session_20261116_201727
